# IBM Applied Data Science Capstone
## Lab 1 — SpaceX API Data Collection

**Goal:** retrieve SpaceX launch records, inspect their structure, and create a reproducible CSV.

This notebook is an independently prepared learning notebook. **Outputs are not pre-populated**; run the cells in order and verify the response before using any results in your report.

### Step 1 — Import packages
`requests` downloads API responses; `pandas` organizes tabular data.

In [ ]:
import requests
import pandas as pd
from pathlib import Path

print("pandas version:", pd.__version__)

### Step 2 — Request SpaceX launch records
The SpaceX v4 API returns JSON records. Check the HTTP status before processing.

In [ ]:
url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(url, timeout=30)
response.raise_for_status()
launches = response.json()
print("Number of launches returned:", len(launches))
print("First record keys:", list(launches[0].keys()) if launches else [])

### Step 3 — Inspect one record
Launch records contain IDs for rockets, payloads, and launchpads. These are references, not descriptive names.

In [ ]:
if launches:
    sample = launches[0]
    for key in ["flight_number", "date_utc", "rocket", "payloads", "launchpad", "success", "cores"]:
        print(key, ":", sample.get(key))

### Step 4 — Build a first DataFrame
We retain source IDs and the first core landing outcome. Later, we will enrich rocket, payload and launchpad information and filter to the required Falcon 9 cohort.

In [ ]:
records = []
for launch in launches:
    cores = launch.get("cores") or []
    first_core = cores[0] if cores else {}
    records.append({
        "FlightNumber": launch.get("flight_number"),
        "Date": launch.get("date_utc"),
        "RocketID": launch.get("rocket"),
        "LaunchpadID": launch.get("launchpad"),
        "PayloadIDs": launch.get("payloads", []),
        "LaunchSuccess": launch.get("success"),
        "LandingSuccess": first_core.get("landing_success"),
        "LandingType": first_core.get("landing_type"),
        "CoreID": first_core.get("core"),
    })
launch_df = pd.DataFrame(records)
launch_df["Date"] = pd.to_datetime(launch_df["Date"], utc=True, errors="coerce")
print("Shape:", launch_df.shape)
display(launch_df.head())

### Step 5 — Basic data quality checks
Missing values are expected in some historical launch records. Do not interpret missing landing success as failure.

In [ ]:
display(launch_df.isna().sum().to_frame("missing_count"))
print("Unique rocket IDs:", launch_df["RocketID"].nunique())
print("Launch date range:", launch_df["Date"].min(), "to", launch_df["Date"].max())

### Step 6 — Save the raw launch table
The raw table is a checkpoint, **not** the final IBM Falcon 9 dataset.

In [ ]:
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "spacex_launches_raw.csv"
launch_df.to_csv(output_path, index=False)
print("Saved:", output_path.resolve())

### What you learned
- **API:** a service interface that provides structured data.
- **JSON:** the nested format returned by the SpaceX API.
- **DataFrame:** pandas tabular structure for analysis.
- **Missing data:** unavailable information that requires careful interpretation.

**Next notebook:** enrich IDs using rocket, payload and launchpad endpoints; restrict to Falcon 9 and build the course-aligned analysis dataset.

**Before GitHub:** run all cells successfully, save the notebook with outputs, and confirm the CSV contains data.